Applied Machine Learning (CSAI2017P) — Lab Assignment 1

Name: Ekta Bhardwaj  
SAP ID: 590022698  
Batch: 5  

## A1. Environment Proof

In [ ]:
import sys
import numpy as np
import pandas as pd
import sklearn
import matplotlib
import seaborn as sns

print("Python       :", sys.version)
print("NumPy        :", np.__version__)
print("Pandas       :", pd.__version__)
print("Scikit-learn :", sklearn.__version__)
print("Matplotlib   :", matplotlib.__version__)
print("Seaborn      :", sns.__version__)

Python       : 3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]
NumPy        : 2.4.4
Pandas       : 3.0.2
Scikit-learn : 1.9.0
Matplotlib   : 3.11.1
Seaborn      : 0.13.2


### Observation (A1)

The environment is reproducible when the required Python and library versions are clearly specified. Pinning versions ensures consistent results across different systems and helps prevent compatibility issues caused by changes between library releases.


In [2]:
from sklearn.datasets import fetch_california_housing

X, y = fetch_california_housing(return_X_y=True, as_frame=True)

df = X.copy()
df["MedHouseVal"] = y

print("Dataset shape:", df.shape)

Dataset shape: (20640, 9)


In [3]:
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB


In [5]:
df.describe()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704,2.068558
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532,1.153956
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000,0.149990
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000,1.196000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000,1.797000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000,2.647250
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000,5.000010


In [6]:
df.isna().sum()

MedInc         0
HouseAge       0
AveRooms       0
AveBedrms      0
Population     0
AveOccup       0
Latitude       0
Longitude      0
MedHouseVal    0
dtype: int64

### Observation

The California Housing dataset contains 20,640 records and 8 numeric features, with `MedHouseVal` as the target variable measured in $100,000 units. The dataset contains no missing values, as all columns have 20,640 non-null entries and `isna().sum()` returns zero for every column. The features are not on the same numerical scale; for example, `Population` ranges from 3 to 35,682, while `MedInc` ranges from 0.4999 to 15.0001. This difference in scale can be important for distance-based algorithms such as KNN.

**A2(c)**  
`Population` appears to be on a very different numerical scale from most other features. Its values range from 3 to 35,682, whereas `MedInc` ranges only from 0.4999 to 15.0001. This large difference in scale suggests that `Population` could have a stronger influence on distance-based methods such as KNN if the features are not scaled.

In [7]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=0
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 16512
Testing samples: 4128


In [8]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error

knn_raw = KNeighborsRegressor(n_neighbors=5)

knn_raw.fit(X_train, y_train)

y_pred_raw = knn_raw.predict(X_test)

mae_raw = mean_absolute_error(y_test, y_pred_raw)

print("Test MAE without scaling:", mae_raw)

Test MAE without scaling: 0.8141586656976745


In [9]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

knn_scaled = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor(n_neighbors=5))
])

knn_scaled.fit(X_train, y_train)

y_pred_scaled = knn_scaled.predict(X_test)

mae_scaled = mean_absolute_error(y_test, y_pred_scaled)

print("Test MAE with scaling:", mae_scaled)

Test MAE with scaling: 0.430758769379845


### Observation (B1)(c)
 
KNN has no learned coefficients; it relies directly on distances between observations to identify the nearest neighbours.  
Without scaling, high-range variables such as `Population` can disproportionately affect these distances and distort neighbourhood selection.  
`StandardScaler` transforms each feature to a comparable scale, allowing KNN to consider their relative patterns more fairly.  
This changes the neighbourhoods used for prediction and reduces the test MAE from **0.8142** to **0.4308**, showing that scaling is crucial for this distance-based model.

**C1(b) Reproducibility Result:**  
The notebook produced identical numerical results after a complete kernel restart and top-to-bottom execution. The raw KNN model returned a test MAE of **0.8141586656976745**, while the scaled pipeline returned **0.430758769379845**. Since neither value changed, the execution was fully reproducible under the same environment and `random_state=0`. This confirms that the reported results are not dependent on the previous kernel state or execution order.